# Laboratorio: Agente RAG con LangChain y OCI Vector Store

En este laboratorio construiremos un agente con **LangChain** capaz de consultar documentos almacenados en un **OCI Generative AI Vector Store**.

## Objetivos

Al finalizar este laboratorio podrás:

- Descubrir y seleccionar documentos desde una carpeta de datos.
- Conectarte a los endpoints compatibles con OpenAI de OCI Generative AI.
- Crear un proyecto y un vector store de OCI Generative AI desde la consola.
- Configurar el identificador del vector store creado para usarlo desde el notebook.
- Cargar e indexar un documento de manera opcional.
- Ejecutar una búsqueda semántica directa.
- Convertir la búsqueda en una tool de LangChain.
- Crear un agente que responda utilizando la base de conocimiento.

---

## Preparación en la consola de OCI

### Crear un proyecto

Antes de ejecutar el notebook, crea los recursos desde la consola de OCI. Selecciona una región donde esté disponible Generative AI y utiliza el mismo *compartment* para el proyecto y el vector store. Esa región debe coincidir con `OCI_REGION` en `.env`.

1. Abre el menú de navegación y entra en **Analytics & AI → Generative AI**.

   ![Ruta de Analytics & AI a Generative AI en la consola de OCI](images/imagen1.png)

2. En el menú lateral de Generative AI, selecciona **Projects**.

   ![Opción Projects en el menú de Generative AI](images/imagen2.png)

3. Comprueba el filtro de *compartment* y selecciona **Create project**. Asigna un nombre reconocible, revisa las opciones de retención de datos y pulsa **Create**.

   ![Página de proyectos con el botón Create project](images/imagen3.png)

4. Abre el proyecto y copia su OCID. Lo usarás como `OCI_GENERATIVE_AI_PROJECT_ID` en `.env`.

Consulta los [detalles de creación de proyectos en la documentación de Oracle](https://docs.oracle.com/en-us/iaas/Content/generative-ai/create-project.htm).


### Crear un vector store

1. En el mismo menú de Generative AI, selecciona **Vector stores**.

   ![Opción Vector stores en el menú de Generative AI](images/imagen4.png)

2. Comprueba que sigues en la misma región y el *compartment* elegido para el proyecto. Selecciona **Create vector store**.

   ![Página de vector stores con el botón Create vector store](images/imagen5.png)

3. Escribe un nombre, confirma el *compartment* y elige **Unstructured data** como tipo de fuente para documentos y búsquedas semánticas. Revisa la política de expiración por inactividad y pulsa **Create**.
4. Abre el vector store creado y copia su identificador en `OCI_VECTOR_STORE_ID` dentro de `.env`. Usa el OCID del *compartment* elegido en `OCI_COMPARTMENT_ID`.

El vector store recién creado estará vacío. Más adelante, la sección de carga opcional muestra cómo indexar un documento desde `data/` con `OCI_UPLOAD_DOCUMENT=true`.

Consulta los [detalles de creación de vector stores en la documentación de Oracle](https://docs.oracle.com/en-us/iaas/Content/generative-ai/create-vector-store.htm).


---

## 1. Preparación del entorno

El laboratorio utiliza autenticación IAM de OCI y carga la configuración desde `.env`. El entorno de Python debe tener instalados `langchain`, `langchain-openai`, `openai`, `oci-openai`, `httpx` y `python-dotenv`.

El archivo `.env` debe contener como mínimo:

```text
OCI_COMPARTMENT_ID=<compartment-ocid>
OCI_GENERATIVE_AI_PROJECT_ID=<generative-ai-project-ocid>
OCI_VECTOR_STORE_ID=<vector-store-id>
```

También puedes definir `OCI_REGION`, `OCI_PROFILE`, `OCI_MODEL_ID` y `OCI_SOURCE_DOCUMENT` para cambiar el comportamiento sin editar las celdas.

In [ ]:
import os
import time
from pathlib import Path

import httpx
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from oci_openai import OciOpenAI, OciUserPrincipalAuth

### Configuración general

Cargamos las variables de entorno y construimos los endpoints. El proyecto, el vector store y los endpoints deben pertenecer a la misma región.

In [ ]:
load_dotenv()

REGION = os.getenv("OCI_REGION", "us-chicago-1")
PROFILE = os.getenv("OCI_PROFILE", "DEFAULT")
MODEL_ID = os.getenv("OCI_MODEL_ID", "openai.gpt-5")
COMPARTMENT_ID = os.environ["OCI_COMPARTMENT_ID"]
PROJECT_ID = os.environ["OCI_GENERATIVE_AI_PROJECT_ID"]
DATA_DIRECTORY = Path("data")
BASE_URL = f"https://inference.generativeai.{REGION}.oci.oraclecloud.com/openai/v1"
CONTROL_PLANE_URL = (
    f"https://generativeai.{REGION}.oci.oraclecloud.com/20231130/openai/v1"
)

---

## 2. Selección del documento

El notebook descubre automáticamente los archivos `.md` dentro de `data/`. Si `OCI_SOURCE_DOCUMENT` no está definida, selecciona el primer documento disponible.

> **Sugerencia para el laboratorio en vivo:** agrega nuevos archivos a `data/` y cambia `OCI_SOURCE_DOCUMENT` para elegir cuál cargar.

In [ ]:
available_documents = sorted(DATA_DIRECTORY.glob("*.md"))
if not available_documents:
    raise FileNotFoundError("No Markdown documents were found in the data directory.")

configured_document = os.getenv("OCI_SOURCE_DOCUMENT")
document_path = (
    DATA_DIRECTORY / configured_document
    if configured_document
    else available_documents[0]
)
if not document_path.is_file():
    raise FileNotFoundError(f"Document not found: {document_path}")

print("Available documents:")
for path in available_documents:
    print(f"- {path.name}")
print(f"Selected document: {document_path.name}")

---

## 3. Creación de los clientes de OCI

OCI separa las operaciones de inferencia y control plane:

- `client` se utiliza para archivos, búsquedas e inferencia.
- `control_plane_client` se utiliza para administrar vector stores.

Ambos clientes utilizan el perfil IAM configurado en `PROFILE`.

In [ ]:
def create_auth():
    return OciUserPrincipalAuth(profile_name=PROFILE)


client = OciOpenAI(
    auth=create_auth(),
    region=REGION,
    compartment_id=COMPARTMENT_ID,
    project=PROJECT_ID,
)

control_plane_client = OciOpenAI(
    auth=create_auth(),
    base_url=CONTROL_PLANE_URL,
    compartment_id=COMPARTMENT_ID,
    project=PROJECT_ID,
)

---

## 4. Selección del vector store

Recuperamos el vector store creado en la consola mediante `OCI_VECTOR_STORE_ID`. Comprueba que su ID esté definido en `.env` y que pertenezca a la región y al compartment configurados.

In [ ]:
VECTOR_STORE_ID = os.environ["OCI_VECTOR_STORE_ID"]

vector_store = control_plane_client.vector_stores.retrieve(
    vector_store_id=VECTOR_STORE_ID
)
print(f"Selected vector store: {vector_store.name} ({vector_store.id})")
print(f"Status: {vector_store.status}")

---

## 5. Carga opcional del documento

La carga está desactivada de manera predeterminada para evitar duplicados al ejecutar el notebook varias veces. Para cargar el documento seleccionado, cambia `UPLOAD_DOCUMENT` a `True` o define `OCI_UPLOAD_DOCUMENT=true`.

El proceso crea un archivo, lo agrega al vector store mediante un batch y espera hasta que la indexación termine.

In [ ]:
UPLOAD_DOCUMENT = os.getenv("OCI_UPLOAD_DOCUMENT", "false").lower() == "true"

if UPLOAD_DOCUMENT:
    with document_path.open("rb") as file_content:
        uploaded_file = client.files.create(
            file=file_content,
            purpose="user_data",
        )

    batch = client.vector_stores.file_batches.create(
        vector_store_id=VECTOR_STORE_ID,
        file_ids=[uploaded_file.id],
        attributes={"source": document_path.name, "format": "markdown"},
    )

    while True:
        batch_status = client.vector_stores.file_batches.retrieve(
            vector_store_id=VECTOR_STORE_ID,
            batch_id=batch.id,
        )
        if batch_status.status in {"completed", "failed", "cancelled"}:
            break
        time.sleep(2)

    print(f"File: {uploaded_file.id}")
    print(f"Batch: {batch.id}")
    print(f"Status: {batch_status.status}")
else:
    print("Document upload skipped.")

---

## 6. Búsqueda directa en la base de conocimiento

Probamos la búsqueda sin utilizar LangChain. La pregunta está en español, pero el código y los nombres de variables permanecen en inglés.

> **Nota:** `rewrite_query` se establece en `False` porque esta opción no está soportada por el endpoint utilizado.

In [ ]:
search_results = client.vector_stores.search(
    vector_store_id=VECTOR_STORE_ID,
    query="¿Cuáles son los temas y condiciones más importantes del documento?",
    max_num_results=3,
    rewrite_query=False,
)

for result in search_results.data:
    print(f"File: {result.filename}")
    print(f"Relevance: {result.score:.3f}")
    print(result.content[0].text[:500])
    print("-" * 80)

---

## 7. Creación de la tool de búsqueda

La tool encapsula la búsqueda en OCI Vector Store. Su descripción ayuda al modelo a decidir cuándo utilizarla y el código elimina fragmentos duplicados antes de devolver el contexto.

In [ ]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search the configured knowledge base for information relevant to a query."""
    results = client.vector_stores.search(
        vector_store_id=VECTOR_STORE_ID,
        query=query,
        max_num_results=10,
        rewrite_query=False,
    )

    chunks = []
    seen_texts = set()
    for result in results.data:
        for content in result.content:
            text = content.text.strip()
            if not text or text in seen_texts:
                continue

            seen_texts.add(text)
            chunks.append(
                "\n".join(
                    [
                        f"Source: {result.filename}",
                        f"Relevance: {result.score:.3f}",
                        text,
                    ]
                )
            )

    if not chunks:
        return "No results were found in the knowledge base."

    return "\n\n---\n\n".join(chunks)

### Prueba aislada de la tool

Invocamos la tool directamente antes de conectarla al agente. Esto permite validar la recuperación de información por separado.

In [ ]:
tool_result = search_knowledge_base.invoke(
    {"query": "¿Cuáles son los requisitos descritos en el documento?"}
)
print(tool_result[:1500])

---

## 8. Configuración del modelo para LangChain

`ChatOpenAI` se conecta al endpoint compatible con OpenAI de OCI. El cliente HTTP firma cada solicitud mediante las credenciales IAM.

In [ ]:
model = ChatOpenAI(
    model=MODEL_ID,
    api_key="not-used",
    base_url=BASE_URL,
    default_headers={
        "OpenAI-Project": PROJECT_ID,
        "opc-compartment-id": COMPARTMENT_ID,
    },
    http_client=httpx.Client(auth=create_auth()),
    store=False,
    max_retries=2,
)

---

## 9. Creación del agente

El agente combina el modelo y la tool. Las instrucciones le indican que debe consultar la base de conocimiento antes de responder, utilizar el idioma del usuario y reconocer cuando no encuentra información suficiente.

In [ ]:
agent = create_agent(
    model=model,
    tools=[search_knowledge_base],
    system_prompt=(
        "You are a knowledge-base assistant. Reply in the same language as "
        "the user. Always call the search_knowledge_base tool before answering "
        "questions about the available documents. Base your answer only on the "
        "retrieved results and clearly say when the knowledge base does not "
        "contain enough information."
    ),
)

### Función auxiliar para hacer preguntas

La función `ask` recibe una pregunta y devuelve únicamente el texto final del agente.

In [ ]:
def ask(question: str) -> str:
    """Run the agent and return its final text response."""
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]}
    )
    return result["messages"][-1].content

---

## 10. Prueba del agente

La pregunta se realiza en español para comprobar que el agente conserva el idioma del usuario. Puedes reemplazarla por cualquier pregunta relacionada con los documentos indexados.

In [ ]:
response = ask(
    "¿Cuáles son los requisitos, costos y condiciones más importantes?"
)
print(response)

---

## Conclusión

El laboratorio separa los datos de la implementación y permite cambiar documentos, vector stores, regiones y modelos mediante configuración.

Como siguientes pasos puedes:

- Agregar más archivos Markdown a `data/`.
- Seleccionar documentos distintos mediante `OCI_SOURCE_DOCUMENT`.
- Utilizar otro vector store mediante `OCI_VECTOR_STORE_ID`.
- Añadir filtros por atributos durante la búsqueda.
- Incorporar nuevas tools o memoria conversacional.